# E6 — Locked test and qualitative figures

Attach the preprocessed BTXRD Dataset and the Output Dataset/version produced by E5. This notebook reads the validation winner from E5, evaluates only BCE+Dice and that winner on test, then writes final tables and qualitative figures. Do not change the selected loss, checkpoint, or threshold here.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import torch
import yaml

REPO_URL = 'https://github.com/lehngoc/BTXRD-LViT.git'
BRANCH = 'model/loss-ablation-normal-fp'
REPO_ROOT = Path('/kaggle/working/BTXRD-LViT')
WORK_ROOT = Path('/kaggle/working/experiments/loss_ablation')

assert torch.cuda.is_available(), 'Enable a T4 GPU in Kaggle Notebook Settings.'
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations', 'PyYAML'], check=True)

In [ ]:
def find_data_root() -> Path:
    suffix = 'data/exports/btxrd_preprocessed/train.csv'
    for csv_path in Path('/kaggle/input').rglob('train.csv'):
        if csv_path.as_posix().endswith(suffix):
            return csv_path.parents[3]
    raise FileNotFoundError('Missing attached BTXRD preprocessed Dataset.')

matches = list(Path('/kaggle/input').rglob('full_loss_ablation_summary.json'))
if not matches:
    raise FileNotFoundError('Attach the Output Dataset/version from E5 full-stage.')

DATA_ROOT = find_data_root()
full_root_input = matches[0].parent
WORK_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(full_root_input, WORK_ROOT / 'full', dirs_exist_ok=True)
summary = json.loads((WORK_ROOT / 'full/full_loss_ablation_summary.json').read_text(encoding='utf-8'))
winner = summary['validation_winner_overall']
test_losses = ['bce_dice_05'] if winner == 'bce_dice_05' else ['bce_dice_05', winner]
print('Validation winner:', winner)
print('Losses evaluated on test:', test_losses)

In [ ]:
RUNTIME_DIR = Path('/kaggle/working/runtime_configs/test')
source_by_loss = {}
for source in (REPO_ROOT / 'configs/loss_ablation').glob('*.yaml'):
    cfg = yaml.safe_load(source.read_text(encoding='utf-8'))
    source_by_loss[cfg['experiment']['loss_id']] = source

def make_evaluation_config(loss_id: str, seed: int) -> Path:
    cfg = yaml.safe_load(source_by_loss[loss_id].read_text(encoding='utf-8'))
    cfg['data']['root_dir'] = str(DATA_ROOT)
    cfg['training']['device'] = 'cuda'
    cfg['training']['num_workers'] = 2
    destination = RUNTIME_DIR / loss_id / f'seed{seed}.yaml'
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    return destination

for loss_id in test_losses:
    for seed in [42, 52, 62, 72, 82]:
        run_dir = WORK_ROOT / 'full' / loss_id / f'seed{seed}'
        checkpoint = run_dir / 'best.pt'
        assert checkpoint.exists(), checkpoint
        runtime_config = make_evaluation_config(loss_id, seed)
        subprocess.run([
            sys.executable, '-m', 'src.training.evaluate_unet',
            '--config', str(runtime_config), '--checkpoint', str(checkpoint),
            '--split', 'test', '--device', 'cuda',
            '--output', str(run_dir / 'test_metrics.json'),
        ], cwd=REPO_ROOT, check=True)

In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.training.aggregate_loss_ablation',
    '--runs-root', str(WORK_ROOT / 'full'), '--stage', 'full',
], cwd=REPO_ROOT, check=True)

for loss_id in test_losses:
    config = make_evaluation_config(loss_id, 42)
    checkpoint = WORK_ROOT / 'full' / loss_id / 'seed42/best.pt'
    subprocess.run([
        sys.executable, '-m', 'src.training.visualize_unet_predictions',
        '--config', str(config), '--checkpoint', str(checkpoint), '--split', 'test',
        '--device', 'cuda', '--output-dir', str(WORK_ROOT / 'qualitative' / loss_id),
    ], cwd=REPO_ROOT, check=True)

archive = shutil.make_archive('/kaggle/working/loss_ablation_final_artifacts', 'gztar',
                              root_dir='/kaggle/working', base_dir='experiments/loss_ablation')
print('Saved archive:', archive)